# !Advertencia!
Este notebook es solo para realizar pruebas no es el Notebook Oficial del Modelo

# Pruebas con TACO

### Dataset 
http://tacodataset.org

In [ ]:
# !pip install kaggle
# !kaggle datasets download -d kneroma/tacotrashdataset
# !unzip tacotrashdataset.zip -d ./data/

In [6]:
import os
import json
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

TACO_TO_TRASHNET = {
    "Glass bottle": "glass", "Broken glass": "glass", "Glass jar": "glass",
    "Normal paper": "paper", "Paper cup": "paper", "Magazine paper": "paper", 
    "Paper bag": "paper", "Receipts": "paper", "Paper wrapper": "paper", "Tissues": "paper",
    "Corrugated carton": "cardboard", "Drink carton": "cardboard", 
    "Egg carton": "cardboard", "Cardboard box": "cardboard", "Cardboard packaging": "cardboard",
    "Clear plastic bottle": "plastic", "Other plastic bottle": "plastic", 
    "Plastic bottle cap": "plastic", "Plastic lid": "plastic", "Plastic film": "plastic", 
    "Plastic bag": "plastic", "Disposable plastic cup": "plastic", "Plastic container": "plastic", 
    "Styrofoam piece": "plastic", "Polypropylene": "plastic", "Six pack rings": "plastic", "Squeezable tube": "plastic",
    "Aluminium foil": "metal", "Aerosol": "metal", "Aluminium food can": "metal", 
    "Drink can": "metal", "Metal bottle cap": "metal", "Scrap metal": "metal", "Metal lid": "metal",
    "Cigarette": "trash", "Food waste": "trash", "Unlabeled litter": "trash", 
    "Garbage bag": "trash", "Blister pack": "trash", "Masks": "trash", "Shoe": "trash"
}

#### Preprocesamiento SIN escala de 224x224

In [8]:
def process_annotation(ann, img_info, categories, taco_dir, output_dir):
    cat_name = categories.get(ann["category_id"])
    target_folder = TACO_TO_TRASHNET.get(cat_name)
    
    if not target_folder:
        return None 

    # Aquí es donde suele fallar TACO. img_info["file_name"] suele ser "batch_1/000001.jpg"
    src_path = os.path.join(taco_dir, img_info["file_name"])
    
    # FIX: Si no existe, imprimimos una alerta para que veas qué ruta está mal construida
    if not os.path.exists(src_path):
        print(f"⚠️ ALERTA: No se encontró la imagen en la ruta: {src_path}")
        return None

    x, y, w, h = [int(v) for v in ann["bbox"]]
    if w <= 0 or h <= 0:
        return None

    try:
        with Image.open(src_path) as img:
            img_w, img_h = img.size
            
            x_start = max(0, x)
            y_start = max(0, y)
            x_end = min(img_w, x + w)
            y_end = min(img_h, y + h)
            
            if x_start >= x_end or y_start >= y_end:
                return None
                
            cropped = img.crop((x_start, y_start, x_end, y_end))
            dest_file = f"taco_{ann['image_id']}_crop_{ann['id']}.jpg"
            dest_path = os.path.join(output_dir, target_folder, dest_file)
            
            cropped.save(dest_path)
            
            return {
                "taco_image_id": ann["image_id"],
                "annotation_id": ann["id"],
                "taco_category": cat_name,
                "target_folder": target_folder,
                "destination_path": dest_path,
                "confidence_score": 1.0, 
                "action": "MOVE_CROP"
            }
            
    except Exception as e:
        print(f"❌ Error procesando {src_path}: {e}")
        return None

#### Preprocesamiento CON escala de 224x224

In [3]:
from PIL import Image, ImageOps

def process_annotation(ann, img_info, categories, taco_dir, output_dir):
    cat_name = categories.get(ann["category_id"])
    target_folder = TACO_TO_TRASHNET.get(cat_name)
    
    if not target_folder:
        return None 

    src_path = os.path.join(taco_dir, img_info["file_name"])
    
    if not os.path.exists(src_path):
        return None

    x, y, w, h = [int(v) for v in ann["bbox"]]
    if w <= 0 or h <= 0:
        return None

    try:
        with Image.open(src_path) as img:
            img_w, img_h = img.size
            
            x_start = max(0, x)
            y_start = max(0, y)
            x_end = min(img_w, x + w)
            y_end = min(img_h, y + h)
            
            if x_start >= x_end or y_start >= y_end:
                return None
                
            # 1. Extraer el recorte original
            cropped = img.crop((x_start, y_start, x_end, y_end))
            
            # 2. Forzar a RGB para evitar problemas al guardar en JPEG
            if cropped.mode in ("RGBA", "P"):
                cropped = cropped.convert("RGB")
                
            # ---------------------------------------------------------
            # 3. TRANSFORMACIÓN PARA MOBILENETV3 (224x224 con Padding)
            # ---------------------------------------------------------
            TARGET_SIZE = (224, 224)
            
            # thumbnail() redimensiona el lado más largo a 224 manteniendo la proporción
            # Usamos LANCZOS porque es el mejor filtro antialiasing para reducción
            cropped.thumbnail(TARGET_SIZE, Image.Resampling.LANCZOS)
            
            # Crear un lienzo negro de 224x224 (fondo negro es común en CNNs)
            final_img = Image.new("RGB", TARGET_SIZE, (0, 0, 0))
            
            # Calcular coordenadas para centrar la imagen redimensionada en el lienzo
            paste_x = (TARGET_SIZE[0] - cropped.width) // 2
            paste_y = (TARGET_SIZE[1] - cropped.height) // 2
            
            # Pegar en el centro
            final_img.paste(cropped, (paste_x, paste_y))
            # ---------------------------------------------------------
            
            dest_file = f"taco_{ann['image_id']}_crop_{ann['id']}.jpg"
            dest_path = os.path.join(output_dir, target_folder, dest_file)
            
            # 4. Guardar la imagen final ya estandarizada
            final_img.save(dest_path, format="JPEG", quality=95)
            
            if not os.path.exists(dest_path):
                print(f"❌ Fallo silencioso: No se guardó el archivo en {dest_path}")
                return None
            
            return {
                "taco_image_id": ann["image_id"],
                "annotation_id": ann["id"],
                "taco_category": cat_name,
                "target_folder": target_folder,
                "destination_path": dest_path,
                "original_size": [w, h],
                "final_size": [224, 224],
                "confidence_score": 1.0, 
                "action": "MOVE_CROP_AND_PAD"
            }
            
    except Exception as e:
        print(f"❌ Error procesando el recorte de {src_path}: {str(e)}")
        return None

In [9]:
def export_taco_to_trashnet_parallel(taco_dir, output_dir, max_workers=8):
    annotations_path = os.path.join(taco_dir, "annotations.json")
    
    # Validar que el JSON exista antes de empezar
    if not os.path.exists(annotations_path):
        print(f"🚨 ERROR CRÍTICO: No se encontró annotations.json en {annotations_path}")
        return
        
    with open(annotations_path, "r") as f:
        data = json.load(f)
        
    categories = {cat["id"]: cat["name"] for cat in data["categories"]}
    images = {img["id"]: img for img in data["images"]}

    for folder in ["glass", "paper", "cardboard", "plastic", "metal", "trash"]:
        os.makedirs(os.path.join(output_dir, folder), exist_ok=True)

    json_logs = []
    
    print(f"🚀 Iniciando procesamiento de {len(data['annotations'])} anotaciones...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for ann in data["annotations"]:
            img_info = images.get(ann["image_id"])
            if img_info:
                future = executor.submit(process_annotation, ann, img_info, categories, taco_dir, output_dir)
                futures.append(future)

        for future in as_completed(futures):
            result = future.result()
            if result:
                json_logs.append(result)

    if not json_logs:
        print("\n⚠️ PROCESO TERMINADO PERO NO SE GENERÓ NINGÚN CROP.")
        print("Revisa los mensajes de ALERTA arriba. Es probable que las rutas de las imágenes no coincidan.")
        return

    log_path = os.path.join(output_dir, "taco_to_trashnet_mapping_log.json")
    with open(log_path, "w") as f:
        json.dump(json_logs, f, indent=2)
        
    print(f"\n✅ Proceso completado con éxito. {len(json_logs)} recortes guardados.")
    print(f"📄 Log JSON guardado en: {log_path}")

In [10]:
taco_dir = "/Users/jairpalma/Downloads/TACO-master/data"
output_dir = "./dataset/TrashNet_Extended"
export_taco_to_trashnet_parallel(taco_dir=taco_dir, output_dir=output_dir, max_workers=10)

🚀 Iniciando procesamiento de 4784 anotaciones...

✅ Proceso completado con éxito. 3482 recortes guardados.
📄 Log JSON guardado en: ./dataset/TrashNet_Extended/taco_to_trashnet_mapping_log.json
